# MODELO PREDICTIVO

quitar call duration -> data leaking

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression
import os

In [2]:
df = pd.read_csv(r'C:\Users\lucia\Documents\Master\TFM\marketing_analysis_fintech\datos\processed_data\FINTECH_LIMPIO.csv', sep=',')

In [8]:
df.columns

Index(['age', 'job_type', 'marital_status', 'education_level',
       'credit_default', 'has_housing_loan', 'has_personal_loan',
       'contact_method', 'contact_month', 'contact_day', 'call_duration',
       'contact_attempts', 'previously_contacted', 'previous_contacts',
       'previous_campaign_outcome', 'employment_variation_rate',
       'consumer_price_index', 'consumer_confidence_index', 'euribor_3m_rate',
       'total_employment', 'call_duration_group', 'is_new_campaign_client',
       'high_contact_attempts', 'subscribed'],
      dtype='object')

In [11]:
df_web = df[["previous_campaign_outcome","contact_method","contact_month","is_new_campaign_client", 'subscribed']]
df_web.to_csv('demo_data', sep=',', index=False)

In [ ]:
df_web

In [3]:
df['subscribed'].value_counts()

subscribed
no     35654
yes     4532
Name: count, dtype: int64

In [4]:
import numpy as np
import pandas as pd
import os
import shutil
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, 
    roc_curve, 
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score
)

def calcula_metricas(metricas, y_test, y_pred, y_prob, fold):
    """
    Calcula métricas de evaluación para un modelo de clasificación binaria
    y las almacena en una lista acumulativa.

    Parámetros
    ----------
    metricas : list of dict
        Lista donde se irán guardando las métricas de cada fold.
    y_test : array-like
        Valores reales (ground truth) del conjunto de test.
    y_pred : array-like
        Predicciones binarias del modelo.
    y_prob : array-like
        Probabilidades predichas para la clase positiva.
    fold : int
        Número del fold actual (iteración de validación cruzada).

    Retorna
    -------
    None
        Modifica la lista `metricas` añadiendo un diccionario con:
        - fold
        - accuracy
        - precision
        - recall
        - f1
        - auc
    """
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    metricas.append({
        "fold": fold,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc
    })


def viz_metricas(y_test, y_pred, y_prob, fold, resultados_dir):
    """
    Genera y guarda visualizaciones de evaluación del modelo:
    matriz de confusión y curva ROC.

    Parámetros
    ----------
    y_test : array-like
        Valores reales del conjunto de test.
    y_pred : array-like
        Predicciones del modelo.
    y_prob : array-like
        Probabilidades de la clase positiva.
    fold : int
        Número del fold actual.
    resultados_dir : str
        Ruta del directorio donde se guardarán las imágenes generadas.

    Retorna
    -------
    None
        Guarda un archivo PNG con:
        - Matriz de confusión
        - Curva ROC

  
    """
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    # MATRIZ DE CONFUSION
    cm = confusion_matrix(y_test, y_pred)
    im = ax[0].imshow(cm, cmap="Blues")
    ax[0].set_title("Matriz de confusión")
    ax[0].set_xlabel("Predicho")
    ax[0].set_ylabel("Real")

    labels = ["No", "Yes"]
    ax[0].set_xticks(np.arange(len(labels)))
    ax[0].set_yticks(np.arange(len(labels)))
    ax[0].set_xticklabels(labels)
    ax[0].set_yticklabels(labels)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax[0].text(j, i, cm[i, j],
                       ha="center", va="center",
                       color="black", fontsize=12)

    fig.colorbar(im, ax=ax[0])

    # CURVA ROC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax[1].plot(fpr, tpr)
    ax[1].plot([0, 1], [0, 1])
    ax[1].set_title("Curva ROC")
    ax[1].set_xlabel("Ratio falso positivo")
    ax[1].set_ylabel("Ratio verdadero positivo")

    filename = os.path.join(resultados_dir, f"metricas_fold_{fold}.png")
    plt.savefig(filename)

    plt.close()



from sklearn.model_selection import GridSearchCV
from sklearn.base import clone
import pandas as pd

def flujo_kfold_nested(y, X, kf_outer, kf_inner, model, param_grid, resultados_dir):
    metricas = []
    mejores_params_por_fold = {}

    for fold, (train_index, test_index) in enumerate(kf_outer.split(X), start=1):
        print(f"\n--- Fold {fold} ---")

        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        # GridSearch con CV interno
        grid = GridSearchCV(
            estimator=clone(model),
            param_grid=param_grid,
            cv=kf_inner,
            scoring="roc_auc",   # puedes cambiar métrica
            n_jobs=-1
        )

        grid.fit(X_train, y_train)

        best_model = grid.best_estimator_
        best_params = grid.best_params_

        print(f"Mejores params fold {fold}: {best_params}")

        mejores_params_por_fold[fold] = best_params

        # Evaluación en test del fold externo
        y_pred = best_model.predict(X_test)
        y_prob = best_model.predict_proba(X_test)[:, 1]

        calcula_metricas(metricas, y_test, y_pred, y_prob, fold)
        viz_metricas(y_test, y_pred, y_prob, fold, resultados_dir)

    df_metricas = pd.DataFrame(metricas)

    print("\nMetricas por fold")
    print(df_metricas)

    print("\nMedia de metricas")
    print(df_metricas.mean())

    print("\nMejores parámetros por fold")
    print(mejores_params_por_fold)

    return mejores_params_por_fold

Ver como funciona el modelo con todos los datos sin hacer ningún tipo de balanceo.

In [5]:
# Creamos una carpeta donde guardar las imagenes de la matriz de confusión y la curva ROC. Si ya esta creada se borra y se vuelve a crear.
resultados_dir = "resultados_modelo"
if os.path.exists(resultados_dir):
    shutil.rmtree(resultados_dir)  
os.makedirs(resultados_dir)

# Quitamos la columna 'Subscribed'
cols = df.columns.drop('subscribed')

# Transformamos que los 'no' sean 0 y los 'yes' sean 1.
y = df['subscribed'].map({'no':0, 'yes':1})
X = pd.get_dummies(df[cols])

# 5-3 cross validation
kf_outter = KFold(n_splits=5, shuffle=True, random_state=42)
kf_inner = KFold(n_splits=3, shuffle=True, random_state=42)

param_grid = [
    {
        "solver": ["liblinear"],
        "penalty": ["l1", "l2"],
        "C": [0.1, 1]
    }
]

model = LogisticRegression()

flujo_kfold_nested(y, X, kf_outter, kf_inner, model, param_grid, resultados_dir)


--- Fold 1 ---
Mejores params fold 1: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}

--- Fold 2 ---
Mejores params fold 2: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}

--- Fold 3 ---


c:\Users\lucia\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Mejores params fold 3: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}

--- Fold 4 ---
Mejores params fold 4: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}

--- Fold 5 ---
Mejores params fold 5: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}

Metricas por fold
   fold  accuracy  precision    recall        f1       auc
0     1  0.909803   0.663669  0.406836  0.504443  0.937812
1     2  0.909917   0.632911  0.448430  0.524934  0.937128
2     3  0.909295   0.680284  0.410944  0.512375  0.936626
3     4  0.909668   0.663248  0.423119  0.516644  0.930701
4     5  0.912530   0.660746  0.420814  0.514167  0.936172

Media de metricas
fold         3.000000
accuracy     0.910242
precision    0.660172
recall       0.422029
f1           0.514513
auc          0.935688
dtype: float64

Mejores parámetros por fold
{1: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}, 2: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}, 3: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}, 4: {'C': 0.1

{1: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'},
 2: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'},
 3: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'},
 4: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'},
 5: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}}

Ver como funciona el modelo habiendo hecho un balenceo de los datos.

In [6]:
resultados_dir = "resultados_modelo_sample"
if os.path.exists(resultados_dir):
    shutil.rmtree(resultados_dir)  
os.makedirs(resultados_dir)   

df_no = df[df['subscribed'] == 'no']
df_si = df[df['subscribed'] == 'yes']
df_no_sample = df_no.sample(n=7000, random_state=1)
df_sample = pd.concat([df_no_sample, df_si])

cols = df_sample.columns.drop('subscribed')

y = df_sample['subscribed'].map({'no':0, 'yes':1})
X = pd.get_dummies(df_sample[cols])

# 5-3 cross validation
kf_outter = KFold(n_splits=5, shuffle=True, random_state=42)
kf_inner = KFold(n_splits=3, shuffle=True, random_state=42)

param_grid = [
    {
        "solver": ["liblinear"],
        "penalty": ["l1", "l2"],
        "C": [0.1, 1]
    },
 
]

model = LogisticRegression()

flujo_kfold_nested(y, X, kf_outter, kf_inner, model, param_grid, resultados_dir)


--- Fold 1 ---


c:\Users\lucia\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Mejores params fold 1: {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'}

--- Fold 2 ---
Mejores params fold 2: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}

--- Fold 3 ---


c:\Users\lucia\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Mejores params fold 3: {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'}

--- Fold 4 ---


c:\Users\lucia\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Mejores params fold 4: {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'}

--- Fold 5 ---


c:\Users\lucia\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Mejores params fold 5: {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'}

Metricas por fold
   fold  accuracy  precision    recall        f1       auc
0     1  0.857391   0.802848  0.831066  0.816713  0.934505
1     2  0.863459   0.836384  0.809524  0.822735  0.940228
2     3  0.867303   0.830168  0.828317  0.829241  0.938654
3     4  0.858196   0.821121  0.825569  0.823339  0.934435
4     5  0.862099   0.825668  0.832794  0.829216  0.937421

Media de metricas
fold         3.000000
accuracy     0.861689
precision    0.823238
recall       0.825454
f1           0.824249
auc          0.937049
dtype: float64

Mejores parámetros por fold
{1: {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'}, 2: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}, 3: {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'}, 4: {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'}, 5: {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'}}


{1: {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'},
 2: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'},
 3: {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'},
 4: {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'},
 5: {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'}}